In [ ]:
### Create a Date Table (Modeling → New Table)
DateTable = 
ADDCOLUMNS(
    CALENDAR(
        MIN('upi_transaction upi_transaction_history'[timestamp]),
        MAX('upi_transaction upi_transaction_history'[timestamp])
    ),
    "Year", YEAR([Date]),
    "Month Number", MONTH([Date]),
    "Month", FORMAT([Date], "MMM"),
    "Year Month", FORMAT([Date], "YYYY-MM"),
    "Quarter", "Q" & FORMAT([Date], "Q")
)



### Create Basic DAX Measures

In [ ]:
### Total Transactions
Total Transactions =
COUNTROWS('upi_transaction upi_transaction_history')

## Total Transaction Amount
Total Transaction Amount =
SUM('upi_transaction upi_transaction_history'[amount])

#### Average Transaction Amount
Average Transaction Amount =
AVERAGE('upi_transaction upi_transaction_history'[amount])

### Successful Transactions
Successful Transactions =
CALCULATE(
    [Total Transactions],
    'upi_transaction upi_transaction_history'[status] = "success"
)

### Failed Transactions

Failed Transactions = 
CALCULATE(
    [Total Transactions],
    'upi_transaction upi_transaction_history'[status] = "failed"
)

### Failure Rate

Failure Rate % =
DIVIDE(
    [Failed Transactions],
    [Total Transactions],
    0
)

### Fraud Transactions

Fraud Transactions = 
CALCULATE(
    [Total Transactions],
    'upi_transaction upi_transaction_history'[fraud_flag] = "True"
)

## Fraud Rate
Fraud Rate % =
DIVIDE(
    [Fraud Transactions],
    [Total Transactions],
    0
)

### Fraud Amount
Fraud Amount =
CALCULATE(
    [Total Transaction Amount],
    'upi_transaction upi_transaction_history'[fraud_flag] = "True"
)

      
### Top Device

Top Device = 
VAR TopDeviceTable =
    TOPN(
        1,
        ALLSELECTED('upi_transaction upi_transaction_history'[device_type]),
        [Total Transaction Amount],
        DESC
    )
RETURN
    CONCATENATEX(
        TopDeviceTable,
        'upi_transaction upi_transaction_history'[device_type],
        ", "
    )

#### Top Merchant

Top Merchant = 
VAR TopMerchant =
    TOPN(
        1,
        FILTER(
            ALLSELECTED('upi_transaction merchant_info'[merchant_id]),
            NOT ISBLANK('upi_transaction merchant_info'[merchant_id])
        ),
        CALCULATE([Total Transaction Amount]),
        DESC,
        'upi_transaction merchant_info'[merchant_id],
        ASC
    )
RETURN
    CONCATENATEX(
        TopMerchant,
        CALCULATE(
            SELECTEDVALUE('upi_transaction merchant_info'[merchant_name])
        ),
        ", "
    )


### Top merchant Sale

Top Merchant Amount = 
VAR TopMerchant =
    TOPN(
        1,
        FILTER(
            ALLSELECTED('upi_transaction merchant_info'[merchant_id]),
            NOT ISBLANK('upi_transaction merchant_info'[merchant_id])
        ),
        CALCULATE([Total Transaction Amount]),
        DESC
    )
RETURN
    CALCULATE(
        [Total Transaction Amount],
        TopMerchant
    )

### Top device Sale     

Top Device Sales =
VAR TopDeviceTable =
    TOPN(
        1,
        ALLSELECTED('upi_transaction upi_transaction_history'[device_type]),
        [Total Transaction Amount],
        DESC
    )
RETURN
    CALCULATE(
        [Total Transaction Amount],
        TopDeviceTable
    )



Region Wise Total Transactions = 
CALCULATE(
    DISTINCTCOUNT('upi_transaction upi_transaction_history'[transaction_id]),
    FILTER(
        'upi_transaction upi_transaction_history',
        NOT ISBLANK('upi_transaction upi_transaction_history'[merchant_id]) &&
        'upi_transaction upi_transaction_history'[merchant_id] <> ""
    )
)

Total Fraud Alerts = 
DISTINCTCOUNT('upi_transaction fraud_alert_history'[alert_id])

High Risk Devices = 
COUNTROWS(
    FILTER(
        VALUES('upi_transaction upi_transaction_history'[device_id]),
        CALCULATE([Total Transactions]) >= 50
            &&
        CALCULATE([Fraud Rate %]) >= 0.05
    )
)

Device Count =
DISTINCTCOUNT('upi_transaction upi_transaction_history'[device_type])

Merchant Total Transactions = 
CALCULATE(
    [Total Transactions],
    FILTER(
        'upi_transaction upi_transaction_history',
        NOT ISBLANK('upi_transaction upi_transaction_history'[merchant_id])
            &&
        TRIM('upi_transaction upi_transaction_history'[merchant_id]) <> ""
    )
)

Merchant Fraud Transactions = 
CALCULATE(
    [Fraud Transactions],
    FILTER(
        'upi_transaction upi_transaction_history',
        NOT ISBLANK('upi_transaction upi_transaction_history'[merchant_id])
            &&
        TRIM('upi_transaction upi_transaction_history'[merchant_id]) <> ""
    )
)

Merchant Fraud Rate = 
DIVIDE(
    [Merchant Fraud Transactions],
    [Merchant Total Transactions],
    0
)

High Risk Merchants = 
COUNTROWS(
    FILTER(
        VALUES('upi_transaction upi_transaction_history'[merchant_id]),
        CALCULATE([Merchant Total Transactions]) >= 50
            &&
        CALCULATE([Merchant Fraud Rate]) >= 0.05
    )
)

High Failure Devices =
COUNTROWS(
    FILTER(
        VALUES('upi_transaction upi_transaction_history'[device_type]),
        CALCULATE([Total Transactions]) >= 50
            &&
        CALCULATE([Failure Rate %]) >= 0.05
    )
)

Merchant Failure Transactions = 
CALCULATE(
    [Failed Transactions],
    FILTER(
        'upi_transaction upi_transaction_history',
        NOT ISBLANK('upi_transaction upi_transaction_history'[merchant_id])
            &&
        TRIM('upi_transaction upi_transaction_history'[merchant_id]) <> ""
    )
)

Merchant Failure Rate = 
DIVIDE(
    [Merchant Failure Transactions],
    [Merchant Total Transactions],
    0
)

High Risk Merchant % = 
DIVIDE(
    [High Risk Merchants],
    [Merchant Total Transactions],
    0
)

### column

Fraud Status = 
IF(
    'upi_transaction upi_transaction_history'[fraud_flag] = "True",
    "Fraud",
    "Non-Fraud"
)